---
title: Getting Started with Qiskit Functions
description: Install the Qiskit Functions Catalog client, authenticate, and learn the common workflow patterns across all vendor functions — loading, running, status checks, result retrieval, and error handling.
---

# Getting Started with Qiskit Functions

The Qiskit Functions Catalog client provides a unified interface for running all functions in the catalog. This page covers installing the client, authenticating, and the core workflow patterns common to all circuit and application functions: loading a function, submitting a job, checking its status, retrieving results, and handling errors.

## Install Qiskit Functions Catalog client

1. To start using Qiskit Functions, install the IBM Qiskit Functions Catalog client:

    ```
    pip install qiskit-ibm-catalog
    ```
1. Retrieve your API key from the [IBM Quantum Platform dashboard](https://quantum.cloud.ibm.com/), and activate your Python virtual environment.  See the [installation instructions](/docs/guides/install-qiskit#local) if you do not already have a virtual environment set up.

    <span id="save-account"></span>**If you are working in a trusted Python environment (such as on a personal laptop or workstation),** use the `save_account()` method to save your credentials locally. ([Skip to the next step](#functions-untrusted) if you are not using a trusted environment, such as a shared or public computer, to authenticate to IBM Quantum Platform.)

    To use `save_account()`, run `python` in your shell, then enter the following:

    ```python
    from qiskit_ibm_catalog import QiskitFunctionsCatalog

    QiskitFunctionsCatalog.save_account(channel="ibm_quantum_platform", token="<your-token>", instance="<instance-crn>")
    ```

    Type `exit()`. From now on, whenever you need to authenticate to the service, you can load your credentials with
    ```python
    from qiskit_ibm_catalog import QiskitFunctionsCatalog
    catalog = QiskitFunctionsCatalog()
    ```

In [ ]:
# Load saved credentials
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

3. <span id="functions-untrusted"></span>**Avoid executing code on an untrusted machine or an external cloud Python environment to minimize security risks.** If you must use an untrusted environment (on, for example, a public computer), change your API key after each use by deleting it on the [IBM Cloud API keys](https://cloud.ibm.com/iam/apikeys) page to reduce risk. Learn more in the [Managing user API keys](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui) topic. To initialize the service in this situation, expand the following section to view code you can use:

    <Accordion>
        <AccordionItem title="Initialize the service in an untrusted environment">

    ```python
    from qiskit_ibm_catalog import QiskitFunctionsCatalog

    # After using the following code, delete your API key on the IBM Quantum Platform home dashboard
    catalog = QiskitFunctionsCatalog(token="<YOUR_API_KEY>") # Use the 44-character API_KEY you created and saved from the IBM Quantum Platform Home dashboard
    ```
      </AccordionItem>
    </Accordion>

    <Admonition type="caution">
      **Protect your API key!** Never include your key in source code, Python script, or notebook file. When sharing code with others, ensure that your API key is not embedded directly within the Python script. Instead, share the script without the key and provide instructions for securely setting it up.

      If you accidentally share your key with someone or include it in version control like Git, immediately revoke your key by deleting it on the [IBM Cloud API keys](https://cloud.ibm.com/iam/apikeys) page to reduce risk. Learn more in the [Managing user API keys](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui) topic.
    </Admonition>

4. After you have authenticated, you can list the functions from the Qiskit Functions Catalog that you have access to:

In [ ]:
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

## Run a function

After a catalog object has been instantiated, you can select a function using `catalog.load(provider/function-name)`:

In [ ]:
ibm_cf = catalog.load("ibm/circuit-function")

Each Qiskit Function has custom inputs, options, and outputs. Check the specific documentation pages for the function you want to run for more information. By default, all users can only run one function job at a time:

In [ ]:
# This cell is hidden from users
# It gets these details programmatically so we can test this notebook
from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit.circuit.random import random_circuit

service = QiskitRuntimeService()
instance = service.active_account()["instance"]
backend_name = service.least_busy().name

circuit = random_circuit(num_qubits=2, depth=2, seed=42)
observable = "Z" * circuit.num_qubits

In [ ]:
job = ibm_cf.run(
    pubs=[(circuit, observable)],
    instance=instance,
    backend_name=backend_name,  # E.g. "ibm_fez"
)

job.job_id

'7f08c9d5-471b-4da2-92e7-4f2cb94c23a8'

## Check job status

<Admonition type="tip">
Currently, the IBM Quantum workloads table only reflects Qiskit Runtime workloads. Use `job.status()` to see your Qiskit Function workload's current status.
</Admonition>

With your Qiskit Function `job_id`, you can check the status of running jobs. This includes the following statuses:

- **`QUEUED`**: The remote program is in the Qiskit Function queue. The queue priority is based on how much you've used Qiskit Functions.
- **`INITIALIZING`**: The remote program is starting; this includes setting up the remote environment and installing dependencies.
- **`RUNNING`**: The program is running. This also includes several more detailed statuses if supported by specific functions
    - **`RUNNING: MAPPING`**": The function is currently mapping your classical inputs to quantum inputs
    - **`RUNNING: OPTIMIZING_FOR_HARDWARE`**": The function is optimizing for the selected QPU. This could include circuit transpilation, QPU characterization, observable backpropagation, and so forth
    - **`RUNNING: WAITING_FOR_QPU`**: The function has submitted a job to Qiskit Runtime, and is waiting in the queue
    - **`RUNNING: EXECUTING_QPU`**: The function has an active Qiskit Runtime job
    - **`RUNNING: POST_PROCESSING`**: The function is post-processing results. This could include error mitigation, mapping quantum results to classical, and so forth
- **`DONE`**: The program is complete, and you can retrieve result data with `job.results()`.
- **`ERROR`**: The program stopped running because of a problem. Use `job.result()` to get the error message.
- **`CANCELED`**: The program was canceled; either by a user, the service, or the server.

In [ ]:
job.status()

'QUEUED'

## Retrieve results

After a program is `DONE`, you can use `job.results()` to fetch the result. This output format varies with each function, so be sure to follow the specific documentation:

In [ ]:
result = job.result()
print(result)

PrimitiveResult([PubResult(data=DataBin(evs=np.ndarray(<shape=(), dtype=float64>), stds=np.ndarray(<shape=(), dtype=float64>), ensemble_standard_error=np.ndarray(<shape=(), dtype=float64>)), metadata={'shots': 4096, 'target_precision': 0.015625, 'circuit_metadata': {}, 'resilience': {}, 'num_randomizations': 32})], metadata={'dynamical_decoupling': {'enable': True, 'sequence_type': 'XX', 'extra_slack_distribution': 'middle', 'scheduling_method': 'alap'}, 'twirling': {'enable_gates': False, 'enable_measure': True, 'num_randomizations': 'auto', 'shots_per_randomization': 'auto', 'interleave_randomizations': True, 'strategy': 'active-accum'}, 'resilience': {'measure_mitigation': True, 'zne_mitigation': False, 'pec_mitigation': False}, 'version': 2})


You can also cancel a job at any time:

In [ ]:
job.stop()

'Job has been stopped.'

## List previous jobs

You can use `jobs()` to list all jobs submitted to Qiskit Functions:

In [ ]:
old_jobs = catalog.jobs()
old_jobs

[<Job | f6c29f49-4d5f-4fff-aca6-2e9a115b9763>,
 <Job | 7f08c9d5-471b-4da2-92e7-4f2cb94c23a8>,
 <Job | 62fe9176-d1e5-467e-b2bd-7a3f3c7be4e5>,
 <Job | af525b2e-16b1-45a1-80bb-dbd94ce30258>,
 <Job | b95a7a57-c1ad-4958-b7ac-953e4e1ee824>,
 <Job | 7bfa33da-0f17-4e67-84b6-f556f7eeb436>,
 <Job | ca46c191-9eb9-4de6-bfa7-b60d7eb29b5e>,
 <Job | 6ac0ba93-3831-43fb-9fb9-760da2225e06>,
 <Job | f0e38071-060d-47e8-988d-9cc1f69358e3>,
 <Job | 629cf110-e490-4675-8a07-f6d298d166b0>]

If you already have the job ID for a certain job, you can retrieve the job with `catalog.get_job_by_id()`:

In [ ]:
# First, get the most recent job that has been executed.
latest_job = old_jobs[0]

# We can also get that same job with get_job_by_id
job_by_id = catalog.get_job_by_id(latest_job.job_id)

# Verify that the job is the same using both retrieval methods.
assert job_by_id.job_id == latest_job.job_id

# Print the job_id for this job.
print(job_by_id.job_id)

f6c29f49-4d5f-4fff-aca6-2e9a115b9763


## Fetch error messages

If a program status is `ERROR`, use `job.error_message()` to fetch the error message as follows:

In [ ]:
job.error_message()

qiskit.exceptions.QiskitError: 'Workflow execution failed -- https://docs.quantum.ibm.com/errors#9999'
